# CSD housing SQL demo

In [265]:
# Connect to the database before any queries 
import duckdb

# The connection puts a lock on the file... 
con = duckdb.connect("data/migration_datasets.duckdb")

print(con.query("SHOW TABLES")) # Print all tables in the database

# ... which MUST be closed before exiting, or all read/writes will be blocked until you kill the responsible process
con.close() 

┌─────────────┐
│    name     │
│   varchar   │
├─────────────┤
│ csd_housing │
└─────────────┘



In [266]:
# Alternatively, use Python's context manager `with` for easier handling
with duckdb.connect("data/migration_datasets.duckdb") as con: 
    print(con.query("SELECT * FROM csd_housing"))

# The `csd_housing` table contains all data from the .csv, just unpivoted for relational data querying

┌───────────┬───────────────┬────────────────────────────────────────────┬──────────────────┬─────────┬────────────┬────────────┬──────────┐
│   city    │  subdivision  │                 geography                  │ immigrant_status │ tenure  │ population │ avg_income │ avg_stir │
│  varchar  │    varchar    │                  varchar                   │     varchar      │ varchar │   double   │   double   │  double  │
├───────────┼───────────────┼────────────────────────────────────────────┼──────────────────┼─────────┼────────────┼────────────┼──────────┤
│ Edmonton  │ Alexander 134 │ Alexander 134 (4811805) IRI 03030 ( 36.0%) │ Both             │ Both    │        0.0 │       NULL │     NULL │
│ Edmonton  │ Alexander 134 │ Alexander 134 (4811805) IRI 03030 ( 36.0%) │ Both             │ Owner   │        0.0 │       NULL │     NULL │
│ Edmonton  │ Alexander 134 │ Alexander 134 (4811805) IRI 03030 ( 36.0%) │ Both             │ Renter  │        0.0 │       NULL │     NULL │
│ Edmonton  │

In [267]:
# Use standard SQL query syntax
with duckdb.connect("data/migration_datasets.duckdb") as con:
    print(
        con.query("SELECT DISTINCT * " \
        "FROM csd_housing " \
        "WHERE geography LIKE '%Canada%'")) # Wildcard expression to see the nation-wide aggregate data

┌─────────┬─────────────┬───────────────────────┬──────────────────┬─────────┬────────────┬────────────┬──────────┐
│  city   │ subdivision │       geography       │ immigrant_status │ tenure  │ population │ avg_income │ avg_stir │
│ varchar │   varchar   │        varchar        │     varchar      │ varchar │   double   │   double   │  double  │
├─────────┼─────────────┼───────────────────────┼──────────────────┼─────────┼────────────┼────────────┼──────────┤
│ NULL    │ NULL        │ Canada 20000 (  4.3%) │ Immigrant        │ Owner   │  7674270.0 │   148000.0 │     21.1 │
│ NULL    │ NULL        │ Canada 20000 (  4.3%) │ Immigrant        │ Both    │ 11324455.0 │   129800.0 │     22.1 │
│ NULL    │ NULL        │ Canada 20000 (  4.3%) │ Non-immigrant    │ Renter  │  6113130.0 │    80800.0 │     23.8 │
│ NULL    │ NULL        │ Canada 20000 (  4.3%) │ Non-immigrant    │ Both    │ 23648635.0 │   131400.0 │     17.7 │
│ NULL    │ NULL        │ Canada 20000 (  4.3%) │ Both             │ Own

In [268]:
with duckdb.connect("data/migration_datasets.duckdb") as con:
    print(
        con.query("SELECT DISTINCT city, subdivision " \
        "FROM csd_housing " \
        "WHERE geography NOT LIKE '%Canada%'")) # Filter out nation-wide data
    # Here we find we have 187 unique subdivisions to work with

┌───────────┬──────────────────────────┐
│   city    │       subdivision        │
│  varchar  │         varchar          │
├───────────┼──────────────────────────┤
│ Edmonton  │ Bon Accord               │
│ Montreal  │ Pointe-Calumet           │
│ Montreal  │ Vaudreuil-sur-le-Lac     │
│ Montreal  │ Vercheres                │
│ Toronto   │ East Gwillimbury         │
│ Vancouver │ Barnston Island 3        │
│ Vancouver │ Lions Bay                │
│ Vancouver │ Musqueam 2               │
│ Edmonton  │ Devon                    │
│ Edmonton  │ Golden Days              │
│    ·      │     ·                    │
│    ·      │     ·                    │
│    ·      │     ·                    │
│ Edmonton  │ Bruderheim               │
│ Edmonton  │ Point Alison             │
│ Montreal  │ Hampstead                │
│ Montreal  │ Repentigny               │
│ Montreal  │ Saint-Amable             │
│ Montreal  │ Saint-Mathieu-de-Beloeil │
│ Toronto   │ Georgina                 │
│ Toronto   │ Mi

In [269]:
# Subdivisions generally lay outside of the named city proper
with duckdb.connect("data/migration_datasets.duckdb") as con:
    print(con.query("SELECT DISTINCT city, subdivision FROM csd_housing WHERE city = 'Toronto' LIMIT 5"))

┌─────────┬─────────────┐
│  city   │ subdivision │
│ varchar │   varchar   │
├─────────┼─────────────┤
│ Toronto │ Milton      │
│ Toronto │ Mono        │
│ Toronto │ Pickering   │
│ Toronto │ Uxbridge    │
│ Toronto │ Newmarket   │
└─────────┴─────────────┘



In [270]:
# We adopt the convention that (subdivision == city) specifies the actual city
with duckdb.connect("data/migration_datasets.duckdb") as con:
    print(con.query("SELECT DISTINCT city, subdivision FROM csd_housing WHERE city = subdivision"))

┌───────────┬─────────────┐
│   city    │ subdivision │
│  varchar  │   varchar   │
├───────────┼─────────────┤
│ Vancouver │ Vancouver   │
│ Montreal  │ Montreal    │
│ Edmonton  │ Edmonton    │
│ Toronto   │ Toronto     │
└───────────┴─────────────┘



In [271]:
# Recalling that our CSD data has three numerical quantities {population, avg_income, avg_stir},
# we can filter for specific conditions very compactly!
with duckdb.connect("data/migration_datasets.duckdb") as con:
    print(con.query("SELECT DISTINCT city, subdivision FROM csd_housing WHERE avg_stir > 30"))
    # 16 subdivisions are diagnosed with brokie... 

┌───────────┬───────────────────┐
│   city    │    subdivision    │
│  varchar  │      varchar      │
├───────────┼───────────────────┤
│ Toronto   │ Markham           │
│ Toronto   │ Oakville          │
│ Toronto   │ Richmond Hill     │
│ Edmonton  │ Calmar            │
│ Vancouver │ Belcarra          │
│ Toronto   │ Milton            │
│ Montreal  │ Sainte-Julie      │
│ Vancouver │ White Rock        │
│ Toronto   │ Vaughan           │
│ Toronto   │ Aurora            │
│ Edmonton  │ Bon Accord        │
│ Toronto   │ East Gwillimbury  │
│ Vancouver │ Lions Bay         │
│ Vancouver │ Metro Vancouver A │
│ Vancouver │ West Vancouver    │
│ Montreal  │ Hudson            │
└───────────┴───────────────────┘
  16 rows             2 columns



In [291]:
# Maybe even some more sophisticated queries comparing owner to renter STIR gaps (reproducing the "cleaned" dataset provided)
with duckdb.connect("data/migration_datasets.duckdb") as con:
    query = """
        SELECT 
            city, 
            subdivision,
            immigrant_status,
            MAX(avg_stir) FILTER (WHERE tenure = 'Owner') AS owner_stir,
            MAX(avg_stir) FILTER (WHERE tenure = 'Renter') AS renter_stir,
            MAX(avg_stir) FILTER (WHERE tenure = 'Renter') - MAX(avg_stir) FILTER (WHERE tenure = 'Owner') AS stir_gap
        FROM csd_housing
        WHERE city IS NOT NULL
        GROUP BY city, subdivision, immigrant_status
        ORDER BY stir_gap DESC
    """
    print(con.query(query))

┌───────────┬───────────────────┬──────────────────┬────────────┬─────────────┬────────────────────┐
│   city    │    subdivision    │ immigrant_status │ owner_stir │ renter_stir │      stir_gap      │
│  varchar  │      varchar      │     varchar      │   double   │   double    │       double       │
├───────────┼───────────────────┼──────────────────┼────────────┼─────────────┼────────────────────┤
│ Vancouver │ Belcarra          │ Both             │       16.5 │        39.5 │               23.0 │
│ Montreal  │ Hudson            │ Immigrant        │       17.8 │        34.5 │               16.7 │
│ Vancouver │ Lions Bay         │ Immigrant        │       20.3 │        36.0 │               15.7 │
│ Vancouver │ Belcarra          │ Non-immigrant    │       15.6 │        31.0 │               15.4 │
│ Vancouver │ Metro Vancouver A │ Non-immigrant    │       17.5 │        32.9 │ 15.399999999999999 │
│ Vancouver │ West Vancouver    │ Non-immigrant    │       16.5 │        31.6 │ 15.10000000

## Aggregate statistic removal test

It doesn't make sense so dw about it

In [272]:
raise NotImplementedError("Do not use.")

NotImplementedError: Do not use.

In [ ]:
# We can recompute aggregate statistics found in the data with appropriate filtering
with duckdb.connect("data/migration_datasets.duckdb") as con:

    # Sum over non-aggregate data
    tbl1 = con.query("SELECT city, SUM(population) " 
        "FROM csd_housing " 
        "WHERE immigrant_status != 'Both' AND tenure != 'Both' " # the 'Both' flags describe aggregate data
        "GROUP BY city " 
        "ORDER BY sum(population) DESC ").df()
    
    # Sum over aggregate data
    tbl2 = con.query("SELECT city, SUM(population) " 
        "FROM csd_housing " 
        "WHERE immigrant_status = 'Both' AND tenure = 'Both' " # this line is different ok. read carefully
        "GROUP BY city " \
        "ORDER BY sum(population) DESC ").df()
    
    print(tbl1)
    print(tbl2)
    
    # Pretty similar! 
    
    # Note the NaN. This corresponds to the nation-wide Canada row. 

In [ ]:
# We can check the data without aggregates ... 
with duckdb.connect("data/migration_datasets.duckdb") as con:
    print(
        con.query("SELECT * "
        "FROM csd_housing "
        "WHERE tenure != 'Both' " \
            "AND immigrant_status != 'Both' " \
            "AND city IS NOT NULL")
        )

In [ ]:
with duckdb.connect("data/migration_datasets.duckdb") as con:
    print(con.query("SELECT * FROM csd_housing"))

In [ ]:
with duckdb.connect("data/migration_datasets.duckdb") as con:
    print(con.query("SELECT * FROM csd_housing_base"))

In [ ]:
raise NotImplementedError("This doesn't work without some moderately annoying weighted sum arithmetic. Do not use.")

# Actually as a sanity check let's recompute the aggregate avg_income and avg_stir as well 
with duckdb.connect("data/migration_datasets.duckdb") as con:
    
    # Compute total avg_income using de-aggregated data
    income_cmp = con.query(
        "SELECT city, sum(avg_income) "
        "FROM csd_housing_base " # the 
        "WHERE city IS NOT NULL "
        "GROUP BY city "
        "ORDER BY sum(avg_income) DESC")
    print(income_cmp)

    # Retrieve total avg_income from the raw, unpivoted data
    income_agg = con.query(
        "SELECT city, sum(avg_income) "
        "FROM csd_housing "
        "WHERE tenure = 'Both' AND immigrant_status = 'Both' AND city IS NOT NULL "
        "GROUP BY city "
        "ORDER BY sum(avg_income) DESC")
    print(income_agg)


In [ ]:
raise RuntimeError("This is buns bro do NOT ues")

# If we're happy, just create a `view` to establish a new interface for the data. 
# No need to rewrite the database
with duckdb.connect("data/migration_datasets.duckdb") as con:
    con.execute("""
        CREATE OR REPLACE VIEW csd_housing_base AS 
        SELECT * 
        FROM csd_housing 
        WHERE tenure != 'Both' 
            AND immigrant_status != 'Both'
    """)

In [ ]:
# Verify that the new view was written correctly
with duckdb.connect("data/migration_datasets.duckdb") as con:
    print(con.query("SHOW TABLES"))

In [ ]:
# Be sure to close your connection or be ready to kill Python processes
con.close()